In [2]:
import numpy as np
import torch


from dataset import load_data, _print_summary
from train_ import get_args

import sys
sys.argv = ["train_.py"]  # use defaults; or add flags e.g. ["train_.py", "--epochs", "50"]

from train_ import get_args
args = get_args()


/home/binal1/miniconda3/envs/pathway-reaction-kg-gpu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
args = get_args()

ctx = load_data(
    data_dir=args.data_dir,
    random_seed=args.seed,
    remove_is_part_of=True,
    embedded_only_ranking=True,
    disjoint_train_ratio=0.2,
    keep_catalyzed_by=False,
    load_ec_embeddings=args.ec_features,
    remove_currency_metabolites=args.remove_currency_metabolites,
    remove_all_metabolites=args.remove_all_metabolites,
    organism_embeddings_path=args.organism_embeddings_path,
    organism_embedding_type=args.organism_embedding_type,
    remove_gene_organism_edges=args.remove_gene_organism_edges,
    remove_organism=args.remove_organism,
    download=args.download,
    print_summary=args.print_dataset_summary,
)

Loading graph from /gpfs/home5/binal1/plantmetbench/data …
  Protein embeddings: 8,445/13,682 nodes covered (61.7%)  dim=960
  Removed 6 is_part_of edge type(s) and Pathway node store
  Removed 3 Organism edge type(s) and Organism node store
Loading taxa split …
  Conversion fingerprints: 19,927/84,470 nodes (23.6%)  dim=3072
  Ranking pool: 8,445/13,682 proteins have embeddings (61.7%)

────────────────────────────────────────────────────────────────────────
Dataset summary
────────────────────────────────────────────────────────────────────────
  Target edge : ('Protein', 'catalyzes', 'Interaction')

Node types:
  GeneProduct   num_nodes=    6,480  x=(6480, 1024)
  Interaction   num_nodes=   84,470  x=(84470, 3072)
  Metabolite    num_nodes=   17,476  x=(17476, 1024)
  Protein       num_nodes=   13,682  x=(13682, 960)
Edge types:
  ('GeneProduct', 'encodes', 'Protein')               num_edges=     2,762
  ('GeneProduct', 'is_about', 'GeneProduct')          num_edges=     4,580
  ('In

This is the number of zero vectors and its percentages in each node type

In [7]:
for nt, x in ctx.train_data.x_dict.items():
    n_zero = (x.abs().sum(dim=1) == 0).sum().item()
    print(f"{nt}: {n_zero}/{x.shape[0]} zero-input rows ({100*n_zero/x.shape[0]:.1f}%)")


GeneProduct: 5126/6480 zero-input rows (79.1%)


Interaction: 70753/84470 zero-input rows (83.8%)
Metabolite: 13277/17476 zero-input rows (76.0%)
Protein: 5237/13682 zero-input rows (38.3%)


Following cell investigates the connections of those zero-vectors

In [8]:
d = ctx.train_data

# ── in/out degree per node, across ALL edge types actually used for message passing ──
node_counts = {nt: d[nt].num_nodes for nt in d.node_types}
in_deg  = {nt: torch.zeros(n, dtype=torch.long) for nt, n in node_counts.items()}
out_deg = {nt: torch.zeros(n, dtype=torch.long) for nt, n in node_counts.items()}

for et in d.edge_types:
    ei = d[et].edge_index
    if ei.shape[1] == 0:
        continue
    src_t, _, dst_t = et
    out_deg[src_t].scatter_add_(0, ei[0], torch.ones(ei.shape[1], dtype=torch.long))
    in_deg[dst_t].scatter_add_(0, ei[1], torch.ones(ei.shape[1], dtype=torch.long))

print(f"{'node_type':<14}{'n_nodes':>10}{'zero_x%':>10}{'in=0 %':>10}{'out=0 %':>10}{'both=0 %':>10}{'mean_in':>10}{'mean_out':>10}")
for nt in d.node_types:
    n = node_counts[nt]
    x = d[nt].x
    n_zero_x = (x.abs().sum(dim=1) == 0).sum().item() if x is not None else -1
    ind, outd = in_deg[nt], out_deg[nt]
    n_in0  = (ind == 0).sum().item()
    n_out0 = (outd == 0).sum().item()
    n_both0 = ((ind == 0) & (outd == 0)).sum().item()
    print(f"{nt:<14}{n:>10,}{100*n_zero_x/n:>9.1f}%{100*n_in0/n:>9.1f}%{100*n_out0/n:>9.1f}%{100*n_both0/n:>9.1f}%{ind.float().mean().item():>10.2f}{outd.float().mean().item():>10.2f}")

# ── break Interaction down by subtype ──────────────────────────────────────
inter = ctx.nodes_df[ctx.nodes_df.node_type == "Interaction"].reset_index(drop=True).copy()
inter["interaction_subtype"] = inter["interaction_subtype"].fillna("")
conv_idx  = torch.tensor(inter.index[inter.interaction_subtype == "Conversion"].tolist())
blank_idx = torch.tensor(inter.index[inter.interaction_subtype == ""].tolist())

print("\nInteraction, split by subtype:")
for label, idx in [("Conversion", conv_idx), ("blank", blank_idx)]:
    ind, outd = in_deg["Interaction"][idx], out_deg["Interaction"][idx]
    n = len(idx)
    n_in0  = (ind == 0).sum().item()
    n_out0 = (outd == 0).sum().item()
    n_both0 = ((ind == 0) & (outd == 0)).sum().item()
    print(f"  {label:<12}{n:>8,} nodes | in=0: {n_in0:>7,} ({100*n_in0/n:.1f}%) | out=0: {n_out0:>7,} ({100*n_out0/n:.1f}%) | both=0 (fully isolated): {n_both0:>7,} ({100*n_both0/n:.1f}%) | mean_in={ind.float().mean():.2f}  mean_out={outd.float().mean():.2f}")

node_type        n_nodes   zero_x%    in=0 %   out=0 %  both=0 %   mean_in  mean_out
GeneProduct        6,480     79.1%      0.1%     57.7%      0.1%      1.15      1.13
Interaction       84,470     83.8%     56.5%     64.8%     29.6%      0.56      1.19
Metabolite        17,476     76.0%      1.4%     82.4%      0.9%      4.22      0.73
Protein           13,682     38.3%      7.3%     72.7%      0.1%      0.93      1.49

Interaction, split by subtype:
  Conversion    19,927 nodes | in=0:  12,962 (65.0%) | out=0:       0 (0.0%) | both=0 (fully isolated):       0 (0.0%) | mean_in=0.88  mean_out=3.62
  blank         64,543 nodes | in=0:  34,803 (53.9%) | out=0:  54,730 (84.8%) | both=0 (fully isolated):  24,990 (38.7%) | mean_in=0.46  mean_out=0.44


In [9]:
ctx.train_data.edge_types

[('GeneProduct', 'encodes', 'Protein'),
 ('GeneProduct', 'is_about', 'GeneProduct'),
 ('Interaction', 'is_about', 'Interaction'),
 ('Interaction', 'participants', 'Interaction'),
 ('Interaction', 'participants', 'Metabolite'),
 ('Interaction', 'source', 'Metabolite'),
 ('Interaction', 'target', 'Interaction'),
 ('Interaction', 'target', 'Metabolite'),
 ('Metabolite', 'is_about', 'Metabolite'),
 ('Protein', 'catalyzes', 'Interaction'),
 ('Protein', 'encoded_by', 'GeneProduct'),
 ('Protein', 'is_about', 'GeneProduct'),
 ('Protein', 'is_about', 'Protein')]

## Metabolites

In [10]:
import sys
sys.path.insert(0, "/home/binal1/plantmetbench/training")
import torch
from dataset import load_data

d = ctx.train_data

met = ctx.nodes_df[ctx.nodes_df.node_type == "Metabolite"].reset_index(drop=True).copy()
met["idx"] = met.index                                  # position == HeteroData row index for Metabolite
is_alias = met["node_id"].str.contains("/Pathway/", regex=False)

alias_idx     = set(met.loc[is_alias,  "idx"].tolist())
canonical_idx = set(met.loc[~is_alias, "idx"].tolist())
print(f"Metabolite nodes: {len(met):,} total | pathway-alias: {len(alias_idx):,} | canonical: {len(canonical_idx):,}")

print(f"\n{'edge_type':<45}{'total':>8}{'alias->X':>10}{'canon->X':>10}{'X->alias':>10}{'X->canon':>10}")
for et in d.edge_types:
    src_t, rel, dst_t = et
    if "Metabolite" not in (src_t, dst_t):
        continue
    ei = d[et].edge_index
    if ei.shape[1] == 0:
        continue
    src, dst = ei[0].tolist(), ei[1].tolist()

    out_alias = out_canon = in_alias = in_canon = 0 
    if src_t == "Metabolite":
        out_alias = sum(1 for s in src if s in alias_idx)
        out_canon = sum(1 for s in src if s in canonical_idx)
    if dst_t == "Metabolite":
        in_alias = sum(1 for dd in dst if dd in alias_idx)
        in_canon = sum(1 for dd in dst if dd in canonical_idx)

    print(f"{str(et):<45}{ei.shape[1]:>8,}{out_alias:>10,}{out_canon:>10,}{in_alias:>10,}{in_canon:>10,}")


Metabolite nodes: 17,476 total | pathway-alias: 12,899 | canonical: 4,577

edge_type                                       total  alias->X  canon->X  X->alias  X->canon
('Interaction', 'participants', 'Metabolite')  34,793         0         0         0    34,793
('Interaction', 'source', 'Metabolite')         7,554         0         0         0     7,554
('Interaction', 'target', 'Metabolite')        18,622         0         0         0    18,622
('Metabolite', 'is_about', 'Metabolite')       12,812         0    12,812    12,812         0


**MAIN FINDING:** Aliases do not contribute to the canonical ones, so it is not changing the model training and does not contribute to the final prediction task, therefore it is redundant to keep them since it adds to the memory and the number of parameters.   

Better to drop them. 

In [11]:
pairs = {}
for rel in ["participants", "source", "target"]:
    et = ("Interaction", rel, "Metabolite")
    ei = d[et].edge_index
    s = set(zip(ei[0].tolist(), ei[1].tolist()))
    pairs[rel] = s
    print(f"{rel:<14}: {len(s):,} unique (Interaction, Metabolite) pairs (raw edge count {ei.shape[1]:,})")

print()
names = list(pairs.keys())
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]
        inter = pairs[a] & pairs[b]
        print(f"overlap --> {a} & {b}: {len(inter):,} shared pairs "
              f"({100*len(inter)/len(pairs[a]):.1f}% of {a}, {100*len(inter)/len(pairs[b]):.1f}% of {b})")

all3 = pairs["participants"] & pairs["source"] & pairs["target"]
union = pairs["participants"] | pairs["source"] | pairs["target"]
print(f"\noverlap --> across all 3: {len(all3):,}")
print(f"union of all 3: {len(union):,}  (sum of individual: {sum(len(p) for p in pairs.values()):,})")

print(f"\nsource subset of participants? {pairs['source'] <= pairs['participants']}")
print(f"target subset of participants? {pairs['target'] <= pairs['participants']}")


participants  : 34,793 unique (Interaction, Metabolite) pairs (raw edge count 34,793)
source        : 7,554 unique (Interaction, Metabolite) pairs (raw edge count 7,554)
target        : 18,622 unique (Interaction, Metabolite) pairs (raw edge count 18,622)

overlap --> participants & source: 7,554 shared pairs (21.7% of participants, 100.0% of source)
overlap --> participants & target: 18,622 shared pairs (53.5% of participants, 100.0% of target)
overlap --> source & target: 97 shared pairs (1.3% of source, 0.5% of target)

overlap --> across all 3: 97
union of all 3: 34,793  (sum of individual: 60,969)

source subset of participants? True
target subset of participants? True


**MAIN FINDING:** There are three edges going from Interaction to Metabolites. Target and source are the subsets of the participants.

If those extra connections don't have any biological meaning, I'd argue, we should drop them (source and target) as well.

In [23]:
inter = ctx.nodes_df[ctx.nodes_df.node_type == "Interaction"].reset_index(drop=True).copy()
inter["interaction_subtype"] = inter["interaction_subtype"].fillna("")
conv_idx_set  = set(inter.index[inter.interaction_subtype == "Conversion"].tolist())
blank_idx_set = set(inter.index[inter.interaction_subtype == ""].tolist())
n_conv, n_blank = len(conv_idx_set), len(blank_idx_set)

def cell(n, denom, width=18):
    return f"{n:,} ({100*n/denom:4.1f}%)".rjust(width)

header = f"{'rel':<15}{'n_edges':>10}{'from Conv':>18}{'from blank':>18}{'uniq Conv src':>20}{'uniq blank src':>20}"
print(header)
print("-" * len(header))
for rel in ["participants", "source", "target"]:
    et = ("Interaction", rel, "Metabolite")
    ei = d[et].edge_index
    n_edges = ei.shape[1]
    src = ei[0].tolist()

    from_conv  = sum(1 for s in src if s in conv_idx_set)
    from_blank = sum(1 for s in src if s in blank_idx_set)

    uniq_src   = set(src)
    uniq_conv  = uniq_src & conv_idx_set
    uniq_blank = uniq_src & blank_idx_set

    row = (
        f"{rel:<15}{n_edges:>10,}"
        f"{cell(from_conv, n_edges)}"
        f"{cell(from_blank, n_edges)}"
        f"{cell(len(uniq_conv), n_conv, 20)}"
        f"{cell(len(uniq_blank), n_blank, 20)}"
    )
    print(row)


rel               n_edges         from Conv        from blank       uniq Conv src      uniq blank src
-----------------------------------------------------------------------------------------------------
participants       34,793    26,062 (74.9%)     8,731 (25.1%)      18,699 (93.8%)       8,731 (13.5%)
source              7,554     7,537 (99.8%)        17 ( 0.2%)       7,537 (37.8%)          17 ( 0.0%)
target             18,622   18,622 (100.0%)         0 ( 0.0%)      18,622 (93.5%)           0 ( 0.0%)


**MAIN FINDING:** 25% of the participants edges between Interaction and Metabolites are coming from zero-vectors (which are not conversions, but 'blank' node types). We were supposed to keep only the Conversions in the Interaction with rather high coverage, but in the current data there is message passing happening from blank interactions to Metabolites. 

I'd argue that those blank nodes (non-conversions) in the Interaction node should be dropped as well. 

## Conversion

Here is the current blank vs Conversion node percentage

In [13]:
inter = ctx.nodes_df[ctx.nodes_df.node_type == "Interaction"].copy()
inter["interaction_subtype"] = inter["interaction_subtype"].fillna("blank")

counts = inter["interaction_subtype"].value_counts()
total = len(inter)

print(f"Total Interaction nodes: {total:,}\n")
for subtype, n in counts.items():
    print(f"  {subtype:<12}: {n:>7,}  ({100*n/total:5.1f}%)")


Total Interaction nodes: 84,470

  blank       :  64,543  ( 76.4%)
  Conversion  :  19,927  ( 23.6%)


In [14]:
inter = ctx.nodes_df[ctx.nodes_df.node_type == "Interaction"].reset_index(drop=True).copy()
inter["interaction_subtype"] = inter["interaction_subtype"].fillna("")
inter_label = ["Conversion" if s == "Conversion" else "blank" for s in inter["interaction_subtype"]]

def label_of(node_type, idx_tensor):
    if node_type == "Interaction":
        return [inter_label[i] for i in idx_tensor.tolist()]
    return [node_type] * len(idx_tensor)

counts, rel_totals = {}, {}
for et in d.edge_types:
    src_t, rel, dst_t = et
    ei = d[et].edge_index
    if ei.shape[1] == 0:
        continue
    rel_totals[et] = ei.shape[1]
    src_labels = label_of(src_t, ei[0])
    dst_labels = label_of(dst_t, ei[1])
    for sl, dl in zip(src_labels, dst_labels):
        key = (sl, rel, dl, et)
        counts[key] = counts.get(key, 0) + 1

rows = [(k, v) for k, v in counts.items() if "Conversion" in (k[0], k[2]) or "blank" in (k[0], k[2])]
rows.sort(key=lambda kv: -kv[1])

print(f"{'src':<12}{'rel':<14}{'dst':<12}{'count':>10}{'% of edge type':>16}")
print("-" * 64)
for (sl, rel, dl, et), n in rows:
    pct_type = 100 * n / rel_totals[et]
    print(f"{sl:<12}{rel:<14}{dl:<12}{n:>10,}{pct_type:>15.1f}%")


src         rel           dst              count  % of edge type
----------------------------------------------------------------
Conversion  participants  Metabolite      26,062           74.9%
Conversion  is_about      blank           19,927           67.0%
Conversion  target        Metabolite      18,622          100.0%
blank       is_about      blank            9,813           33.0%
blank       participants  Conversion       9,813          100.0%
blank       participants  Metabolite       8,731           25.1%
Protein     catalyzes     Conversion       7,655          100.0%
Conversion  source        Metabolite       7,537           99.8%
blank       source        Metabolite          17            0.2%
blank       target        Conversion          17          100.0%


**MAIN FINDING** Here, we again see 25% (see row 6) of the edge (Interaction participants Metabolites) are coming from blank nodes.

Also blank nodes send messages to the Conversions through (Metabolite participants Metabolite) edges which is fully from blank to Conversion (see row 5).

Given those, I'd say we should drop those blank (non-Conversion) nodes from the data.

In [24]:
pairs = {}
for rel in ["is_about", "participants", "target"]:
    et = ("Interaction", rel, "Interaction")
    ei = d[et].edge_index
    s = set(zip(ei[0].tolist(), ei[1].tolist()))
    pairs[rel] = s
    print(f"{rel:<14}: {len(s):,} unique (Interaction, Interaction) pairs (raw edge count {ei.shape[1]:,})")

print()
names = list(pairs.keys())
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]
        ov = pairs[a] & pairs[b]
        print(f"overlap --> {a} & {b}: {len(ov):,} shared pairs "
              f"({100*len(ov)/len(pairs[a]):.1f}% of {a}, {100*len(ov)/len(pairs[b]):.1f}% of {b})")

all3 = pairs["is_about"] & pairs["participants"] & pairs["target"]
union = pairs["is_about"] | pairs["participants"] | pairs["target"]
print(f"\noverlap --> across all 3: {len(all3):,}")
print(f"union of all 3: {len(union):,}  (sum of individual: {sum(len(p) for p in pairs.values()):,})")

print(f"\nparticipants subset of is_about? {pairs['participants'] <= pairs['is_about']}")
print(f"target subset of is_about?       {pairs['target'] <= pairs['is_about']}")
print(f"target subset of participants?   {pairs['target'] <= pairs['participants']}")


is_about      : 29,740 unique (Interaction, Interaction) pairs (raw edge count 29,740)
participants  : 9,813 unique (Interaction, Interaction) pairs (raw edge count 9,813)
target        : 17 unique (Interaction, Interaction) pairs (raw edge count 17)

overlap --> is_about & participants: 0 shared pairs (0.0% of is_about, 0.0% of participants)
overlap --> is_about & target: 0 shared pairs (0.0% of is_about, 0.0% of target)
overlap --> participants & target: 17 shared pairs (0.2% of participants, 100.0% of target)

overlap --> across all 3: 0
union of all 3: 39,553  (sum of individual: 39,570)

participants subset of is_about? False
target subset of is_about?       False
target subset of participants?   True


**MAIN FINDING:** Here, you can see target is the subset of the participants, so if it doesn't represent a different biological connection between conversion and metabolites, I'd say it also needs to be dropped, since those connections are already represented through participants edge.

## GeneProduct

In [16]:

gp = ctx.nodes_df[ctx.nodes_df.node_type == "GeneProduct"].reset_index(drop=True).copy()
gp["idx"] = gp.index
is_alias = gp["node_id"].str.contains("/Pathway/", regex=False)
alias_idx     = set(gp.loc[is_alias, "idx"].tolist())
canonical_idx = set(gp.loc[~is_alias, "idx"].tolist())
print(f"GeneProduct nodes: {len(gp):,} total | pathway-alias: {len(alias_idx):,} | canonical: {len(canonical_idx):,}")

x = d["GeneProduct"].x
is_zero = (x.abs().sum(dim=1) == 0)
for label, idx_set in [("alias", alias_idx), ("canonical", canonical_idx)]:
    idx = torch.tensor(sorted(idx_set), dtype=torch.long)
    z = is_zero[idx]
    print(f"  {label:<10}: zero-x {z.sum().item():,}/{len(idx):,} ({100*z.sum().item()/len(idx):.1f}%)")

print(f"\n{'edge_type':<45}{'total':>8}{'alias->X':>10}{'canon->X':>10}{'X->alias':>10}{'X->canon':>10}")
for et in d.edge_types:
    src_t, rel, dst_t = et
    if "GeneProduct" not in (src_t, dst_t):
        continue
    ei = d[et].edge_index
    if ei.shape[1] == 0:
        continue
    src, dst = ei[0].tolist(), ei[1].tolist()

    out_alias = out_canon = in_alias = in_canon = 0
    if src_t == "GeneProduct":
        out_alias = sum(1 for s in src if s in alias_idx)
        out_canon = sum(1 for s in src if s in canonical_idx)
    if dst_t == "GeneProduct":
        in_alias = sum(1 for dd in dst if dd in alias_idx)
        in_canon = sum(1 for dd in dst if dd in canonical_idx)

    print(f"{str(et):<45}{ei.shape[1]:>8,}{out_alias:>10,}{out_canon:>10,}{in_alias:>10,}{in_canon:>10,}")


GeneProduct nodes: 6,480 total | pathway-alias: 3,739 | canonical: 2,741
  alias     : zero-x 3,739/3,739 (100.0%)
  canonical : zero-x 1,387/2,741 (50.6%)

edge_type                                       total  alias->X  canon->X  X->alias  X->canon
('GeneProduct', 'encodes', 'Protein')           2,762         0     2,762         0         0
('GeneProduct', 'is_about', 'GeneProduct')      4,580         0     4,580     3,650       930
('Protein', 'encoded_by', 'GeneProduct')        2,762         0         0         0     2,762
('Protein', 'is_about', 'GeneProduct')             83         0         0        83         0


**MAIN FINDING:** Similar to the Metabolites, GeneProduct has aliases as well, and those also don't contribute to the canonical GeneProducts, but only receives message from them. This means the aliases do not contribute to the main task.

Aliases are redundant, and we can just drop them as well to save memory and number of parameters, and also evaluate the models more accurately. 

## FINAL EDIT

Given the previous cell results, we can drop the aliases in the GeneProduct and Metabolites, as well as the non-Conversion Interaction nodes.

In [ ]:
d = ctx.train_data.clone()

def keep_mask_non_alias(node_type):
    """True = keep (canonical, non-pathway-alias id)."""
    sub = ctx.nodes_df[ctx.nodes_df.node_type == node_type].reset_index(drop=True)
    is_alias = sub["node_id"].str.contains("/Pathway/", regex=False)
    return torch.tensor((~is_alias).to_numpy(), dtype=torch.bool)

inter = ctx.nodes_df[ctx.nodes_df.node_type == "Interaction"].reset_index(drop=True)
keep_interaction = torch.tensor(
    (inter["interaction_subtype"].fillna("") == "Conversion").to_numpy(), dtype=torch.bool
)

# Drop the aliases and keep the conversions only
subset_dict = {
    "Metabolite":  keep_mask_non_alias("Metabolite"),   # 100% zero-x, safe to drop
    "GeneProduct": keep_mask_non_alias("GeneProduct"),  # 100% zero-x, safe to drop
    "Interaction": keep_interaction,                     # drop blank-subtype
}

d_clean = d.subgraph(subset_dict)

print("Before -> after:")
for nt in d.node_types:
    before = d[nt].num_nodes
    after  = d_clean[nt].num_nodes if nt in d_clean.node_types else 0
    print(f"  {nt:<12}: {before:>7,} -> {after:>7,}")

print("\nEdge types remaining (non-empty):")
for et in d_clean.edge_types:
    n = d_clean[et].edge_index.shape[1]
    if n > 0:
        print(f"  {et}: {n:,}")


Before -> after:
  GeneProduct :   6,480 ->   2,741
  Interaction :  84,470 ->  19,927
  Metabolite  :  17,476 ->   4,577
  Protein     :  13,682 ->  13,682

Edge types remaining (non-empty):
  ('GeneProduct', 'encodes', 'Protein'): 2,762
  ('GeneProduct', 'is_about', 'GeneProduct'): 930
  ('Interaction', 'participants', 'Metabolite'): 26,062
  ('Interaction', 'source', 'Metabolite'): 7,537
  ('Interaction', 'target', 'Metabolite'): 18,622
  ('Protein', 'catalyzes', 'Interaction'): 7,655
  ('Protein', 'encoded_by', 'GeneProduct'): 2,762
  ('Protein', 'is_about', 'Protein'): 9,932


Above, you can see the number of nodes and the edge types left after cleaning. Following edges are also dropped since they were either connected to the aliases or non-Conversion interactions:
- (Metabolites is_about Metabolites)
- (Interaction target Interaction)
- (Interaction is_about Interaction)
- (Interaction participants Interaction)
- (Protein is_about GeneProduct)

See the following cell for confirmation. A new file build_clean_ctx.py is added to make those changes to the train, val and test data. 

In [31]:
# ── Metabolite categories: alias vs canonical ───────────────────────────────
met = ctx.nodes_df[ctx.nodes_df.node_type == "Metabolite"].reset_index(drop=True).copy()
met["idx"] = met.index
is_alias_met = met["node_id"].str.contains("/Pathway/", regex=False)
met_cat = {i: ("alias" if alias else "canonical") for i, alias in zip(met["idx"], is_alias_met)}

print("=== (Metabolite, is_about, Metabolite) ===")
ei = d[("Metabolite", "is_about", "Metabolite")].edge_index
src, dst = ei[0].tolist(), ei[1].tolist()
counts = {}
for s, t in zip(src, dst):
    key = (met_cat[s], met_cat[t])
    counts[key] = counts.get(key, 0) + 1
total = len(src)
for (s_cat, d_cat), n in sorted(counts.items(), key=lambda kv: -kv[1]):
    print(f"  {s_cat:<10} -> {d_cat:<10}: {n:>7,}  ({100*n/total:.1f}%)")

# ── Interaction categories: Conversion vs blank ─────────────────────────────
inter = ctx.nodes_df[ctx.nodes_df.node_type == "Interaction"].reset_index(drop=True).copy()
inter["idx"] = inter.index
inter["interaction_subtype"] = inter["interaction_subtype"].fillna("")
inter_cat = {i: ("Conversion" if sub == "Conversion" else "blank") for i, sub in zip(inter["idx"], inter["interaction_subtype"])}

for rel in ["is_about", "participants", "target"]:
    et = ("Interaction", rel, "Interaction")
    print(f"\n=== {et} ===")
    ei = d[et].edge_index
    if ei.shape[1] == 0:
        print("  (no edges)")
        continue
    src, dst = ei[0].tolist(), ei[1].tolist()
    counts = {}
    for s, t in zip(src, dst):
        key = (inter_cat[s], inter_cat[t])
        counts[key] = counts.get(key, 0) + 1
    total = len(src)
    for (s_cat, d_cat), n in sorted(counts.items(), key=lambda kv: -kv[1]):
        print(f"  {s_cat:<10} -> {d_cat:<10}: {n:>7,}  ({100*n/total:.1f}%)")

# ── GeneProduct categories: alias vs canonical ──────────────────────────────
gp = ctx.nodes_df[ctx.nodes_df.node_type == "GeneProduct"].reset_index(drop=True).copy()
gp["idx"] = gp.index
is_alias_gp = gp["node_id"].str.contains("/Pathway/", regex=False)
gp_cat = {i: ("alias" if alias else "canonical") for i, alias in zip(gp["idx"], is_alias_gp)}

print("\n=== (Protein, is_about, GeneProduct) ===")
et = ("Protein", "is_about", "GeneProduct")
ei = d[et].edge_index
if ei.shape[1] == 0:
    print("  (no edges)")
else:
    src, dst = ei[0].tolist(), ei[1].tolist()
    counts = {}
    for s, t in zip(src, dst):
        key = gp_cat[t]  # Protein has no subtype/alias distinction, only the GeneProduct dst matters here
        counts[key] = counts.get(key, 0) + 1
    total = len(src)
    for d_cat, n in sorted(counts.items(), key=lambda kv: -kv[1]):
        print(f"  Protein -> {d_cat:<10}: {n:>7,}  ({100*n/total:.1f}%)")



=== (Metabolite, is_about, Metabolite) ===
  canonical  -> alias     :  12,812  (100.0%)

=== ('Interaction', 'is_about', 'Interaction') ===
  Conversion -> blank     :  19,927  (67.0%)
  blank      -> blank     :   9,813  (33.0%)

=== ('Interaction', 'participants', 'Interaction') ===
  blank      -> Conversion:   9,813  (100.0%)

=== ('Interaction', 'target', 'Interaction') ===
  blank      -> Conversion:      17  (100.0%)

=== (Protein, is_about, GeneProduct) ===
  Protein -> alias     :      83  (100.0%)


## Train vs Val vs Test


Here I check the overlap between train, val and test set

In [35]:

TARGET_EDGE = ctx.target_edge

for nt in ctx.train_data.node_types:
    same_tv = torch.equal(ctx.train_data[nt].x, ctx.val_data[nt].x)
    same_tt = torch.equal(ctx.train_data[nt].x, ctx.test_data[nt].x)
    print(f"{nt}: train==val? {same_tv}  train==test? {same_tt}")

print()

edge_types = ctx.train_data.edge_types
print(f"{'edge_type':<45}{'train==val':>12}{'train==test':>13}")
for et in edge_types:
    if et == TARGET_EDGE:
        continue
    same_tv = torch.equal(ctx.train_data[et].edge_index, ctx.val_data[et].edge_index)
    same_tt = torch.equal(ctx.train_data[et].edge_index, ctx.test_data[et].edge_index)
    print(f"{str(et):<45}{str(same_tv):>12}{str(same_tt):>13}")


GeneProduct: train==val? True  train==test? True
Interaction: train==val? True  train==test? True
Metabolite: train==val? True  train==test? True
Protein: train==val? True  train==test? True

edge_type                                      train==val  train==test
('GeneProduct', 'encodes', 'Protein')                True         True
('GeneProduct', 'is_about', 'GeneProduct')           True         True
('Interaction', 'is_about', 'Interaction')           True         True
('Interaction', 'participants', 'Interaction')        True         True
('Interaction', 'participants', 'Metabolite')        True         True
('Interaction', 'source', 'Metabolite')              True         True
('Interaction', 'target', 'Interaction')             True         True
('Interaction', 'target', 'Metabolite')              True         True
('Metabolite', 'is_about', 'Metabolite')             True         True
('Protein', 'encoded_by', 'GeneProduct')             True         True
('Protein', 'is_about', 'G

**MAIN FINDING:** Train, val and test have identical node features and edge connections (other than the target edge)

Below, you can find the distribution of the target edge and overlap among the different sets. 

In [21]:

splits = {"train": ctx.train_data, "val": ctx.val_data, "test": ctx.test_data}
TARGET_EDGE = ctx.target_edge

print(f"Target edge: {TARGET_EDGE}\n")

print("Supervision edge counts (edge_label_index / edge_label):")
info = {}
for name, data in splits.items():
    eli = data[TARGET_EDGE].edge_label_index
    lbl = data[TARGET_EDGE].edge_label
    pos_mask = lbl == 1
    neg_mask = lbl == 0
    n_pos, n_neg = int(pos_mask.sum()), int(neg_mask.sum())
    pos_pairs = set(zip(eli[0, pos_mask].tolist(), eli[1, pos_mask].tolist()))
    neg_pairs = set(zip(eli[0, neg_mask].tolist(), eli[1, neg_mask].tolist()))
    pos_prot  = set(eli[0, pos_mask].tolist())
    pos_inter = set(eli[1, pos_mask].tolist())
    info[name] = dict(pos_pairs=pos_pairs, neg_pairs=neg_pairs, pos_prot=pos_prot, pos_inter=pos_inter)
    print(f"  {name:<6}: {eli.shape[1]:>6,} total  ({n_pos:,} pos / {n_neg:,} neg)  "
          f"unique proteins={len(pos_prot):,}  unique interactions={len(pos_inter):,}")

print("\nOverlap of exact (protein, interaction) PAIRS between splits:")
names = list(splits.keys())
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]
        pp = info[a]["pos_pairs"] & info[b]["pos_pairs"]
        nn = info[a]["neg_pairs"] & info[b]["neg_pairs"]
        pn = info[a]["pos_pairs"] & info[b]["neg_pairs"]
        print(f"  {a} vs {b}: positive∩positive={len(pp)}  negative∩negative={len(nn)}  "
              f"positive(a)∩negative(b)={len(pn)}")

print("\nOverlap of unique PROTEIN nodes appearing in positive supervision:")
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]
        ov = info[a]["pos_prot"] & info[b]["pos_prot"]
        print(f"  {a} ∩ {b}: {len(ov)} shared proteins "
              f"({100*len(ov)/len(info[a]['pos_prot']):.1f}% of {a}'s, "
              f"{100*len(ov)/len(info[b]['pos_prot']):.1f}% of {b}'s)")

print("\nOverlap of unique INTERACTION (reaction) nodes appearing in positive supervision:")
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        a, b = names[i], names[j]
        ov = info[a]["pos_inter"] & info[b]["pos_inter"]
        print(f"  {a} ∩ {b}: {len(ov)} shared reactions "
              f"({100*len(ov)/len(info[a]['pos_inter']):.1f}% of {a}'s, "
              f"{100*len(ov)/len(info[b]['pos_inter']):.1f}% of {b}'s)")


Target edge: ('Protein', 'catalyzes', 'Interaction')

Supervision edge counts (edge_label_index / edge_label):
  train :  3,828 total  (1,914 pos / 1,914 neg)  unique proteins=1,176  unique interactions=1,336
  val   :  2,422 total  (1,211 pos / 1,211 neg)  unique proteins=312  unique interactions=897
  test  :  2,400 total  (1,200 pos / 1,200 neg)  unique proteins=357  unique interactions=757

Overlap of exact (protein, interaction) PAIRS between splits:
  train vs val: positive∩positive=0  negative∩negative=0  positive(a)∩negative(b)=0
  train vs test: positive∩positive=0  negative∩negative=0  positive(a)∩negative(b)=0
  val vs test: positive∩positive=0  negative∩negative=0  positive(a)∩negative(b)=0

Overlap of unique PROTEIN nodes appearing in positive supervision:
  train ∩ val: 0 shared proteins (0.0% of train's, 0.0% of val's)
  train ∩ test: 0 shared proteins (0.0% of train's, 0.0% of test's)
  val ∩ test: 0 shared proteins (0.0% of val's, 0.0% of test's)

Overlap of unique INT